Load the dataset

In [3]:
import pprint
import evaluate
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

In [4]:
ds = load_dataset(
    'bitext/Bitext-customer-support-llm-chatbot-training-dataset',
    split="train")

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\datasets--bitext--Bitext-customer-support-llm-chatbot-training-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xe

In [6]:
print(ds.column_names)
print(ds.shape)

['flags', 'instruction', 'category', 'intent', 'response']
(26872, 5)


In [7]:
pprint.pprint(ds[0])
 

{'category': 'ORDER',
 'flags': 'B',
 'instruction': 'question about cancelling order {{Order Number}}',
 'intent': 'cancel_order',
 'response': "I've understood you have a question regarding canceling order "
             "{{Order Number}}, and I'm here to provide you with the "
             'information you need. Please go ahead and ask your question, and '
             "I'll do my best to assist you."}


Filter: small train set + a held-out evaluation set

In [8]:
first_thousand_points = ds[:1000]
train_ds = Dataset.from_dict(first_thousand_points)
 
evaluation_dataset = Dataset.from_dict(ds[1000:1020])

Preprocess: merge instruction + response into one text field

In [9]:
def merge_example(row):
    row['conversation'] = f"Query: {row['instruction']}\nResponse: {row['response']}"
    return row
 
train_ds = train_ds.map(merge_example)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 12109.35 examples/s]


In [10]:
print(train_ds[0]['conversation'])

Query: question about cancelling order {{Order Number}}
Response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


Load the model and tokenizer with Auto classes 

In [11]:
model_name = "Maykeye/TinyLLama-v0"
 
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--Maykeye--TinyLLama-v0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back

Evaluation helpers (ROUGE)

In [12]:
def generate_predictions_and_reference(dataset):
    predictions = []
    references = []
    for row in dataset:
        # same "Query: ... Response:" format the model is trained on
        prompt = f"Query: {row['instruction']}\nResponse:"
        inputs = tokenizer.encode(prompt, return_tensors="pt")
        outputs = model.generate(
            inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
        decoded_outputs = tokenizer.decode(
            outputs[0, inputs.shape[1]:], skip_special_tokens=True)
        references += [row["response"]]
        predictions += [decoded_outputs]
    return references, predictions

In [19]:
rouge = evaluate.load('rouge')

Evaluate BEFORE fine-tuning (baseline)

In [20]:
references, predictions = generate_predictions_and_reference(evaluation_dataset)
results_before = rouge.compute(predictions=predictions, references=references)
print("No fine-tuning:", results_before)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


No fine-tuning: {'rouge1': np.float64(0.14598638683173862), 'rouge2': np.float64(0.007485252824627657), 'rougeL': np.float64(0.11052693531030694), 'rougeLsum': np.float64(0.10909057936987201)}


Training arguments (CPU)

In [21]:
training_arguments = TrainingArguments(
    output_dir="./tiny_finetuned",
    per_device_train_batch_size=1,
    learning_rate=2e-3,
    max_grad_norm=0.3,
    max_steps=200,
    gradient_accumulation_steps=2,
    save_steps=10,
    logging_steps=10,
    use_cpu=True,          # force CPU
    report_to="none",      
)

Set up training with SFTTrainer

In [22]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='conversation',
    max_seq_length=250,
    args=training_arguments
)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 3765.87 examples/s]
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:318: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


train

In [23]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,6.449400
20,4.364800
30,3.600200
40,2.828900
50,2.421200
60,1.939800
70,1.545600
80,1.396400
90,1.454900
100,1.560700


TrainOutput(global_step=200, training_loss=1.9582638263702392, metrics={'train_runtime': 62.6834, 'train_samples_per_second': 6.381, 'train_steps_per_second': 3.191, 'total_flos': 1441641262464.0, 'train_loss': 1.9582638263702392, 'epoch': 0.4})

Evaluate AFTER fine-tuning

In [24]:
references, predictions = generate_predictions_and_reference(evaluation_dataset)
results_after = rouge.compute(predictions=predictions, references=references)
print("Fine-tuned:", results_after)
 
print("\nSample output after fine-tuning:")
print("Query:   ", evaluation_dataset[0]["instruction"])
print("Model:   ", predictions[0])
print("Expected:", references[0])

Fine-tuned: {'rouge1': np.float64(0.3231498152787994), 'rouge2': np.float64(0.12208516859436463), 'rougeL': np.float64(0.2424025324044268), 'rougeLsum': np.float64(0.25450303820187326)}

Sample output after fine-tuning:
Query:    want help adding an item to order {{Order Number}}
Model:    I've realized that you're seeking assistance with canceling your purchase with the order number {{Order Number}}. I apologize for any inconvenience caused. To cancel your purchase, please follow these steps:

1. Sign in to Your
Expected: Thank you for getting in touch to us for assistance with adding an item to your order. We understand the importance of getting your order just right. To help you with this, could you please provide us with the details of the item you would like to add? By having this information, we can ensure that your order is complete and meets your expectations. We appreciate your cooperation and look forward to assisting you further.


Save the fine-tuned model

In [25]:
trainer.save_model("./tiny_finetuned/final")
tokenizer.save_pretrained("./tiny_finetuned/final")


('./tiny_finetuned/final\\tokenizer_config.json',
 './tiny_finetuned/final\\special_tokens_map.json',
 './tiny_finetuned/final\\tokenizer.model',
 './tiny_finetuned/final\\added_tokens.json',
 './tiny_finetuned/final\\tokenizer.json')